# 1. Загрузка необходимых библиотек

In [ ]:
import os

# TensorFlow не ставится на Python 3.14 — тогда берём Keras 3 + PyTorch.
os.environ.setdefault("KERAS_BACKEND", "torch")

import numpy as np
import matplotlib.pyplot as plt
import keras
from keras.models import Sequential
from keras.layers import (
    Input, Conv2D, MaxPooling2D, Flatten, Dense, Dropout,
    RandomFlip, RandomRotation, RandomZoom,
)
from keras.optimizers import Adam
from keras.utils import load_img, img_to_array
from sklearn.model_selection import train_test_split

print("Keras:", keras.__version__, "| backend:", keras.backend.backend())

# 2. Установка параметров

In [ ]:
IMG_SIZE = 128
BATCH_SIZE = 32
EPOCHS = 10
MAX_PER_CLASS = 200

# 3. Скачивание необходимых данных и создание списка категорий

In [ ]:
import kagglehub
path = kagglehub.dataset_download("misrakahmed/vegetable-image-dataset")
print("Path to dataset files:", path)

In [ ]:
train_dir = os.path.join(path, "Vegetable Images", "train")
categories = sorted(
    name for name in os.listdir(train_dir)
    if os.path.isdir(os.path.join(train_dir, name)) and not name.startswith(".")
)
print(f"Категории овощей: {categories}")
print(f"Число классов: {len(categories)}")

In [ ]:
def load_data(folder, img_size, max_per_class=MAX_PER_CLASS):
    """Загружает не больше max_per_class фото на класс, сразу в нужный размер."""
    images, labels = [], []
    for label, category in enumerate(categories):
        class_dir = os.path.join(folder, category)
        files = [
            name for name in os.listdir(class_dir)
            if not name.startswith(".") and name.lower().endswith((".jpg", ".jpeg", ".png"))
        ][:max_per_class]
        for name in files:
            img = load_img(os.path.join(class_dir, name), target_size=(img_size, img_size))
            images.append(img_to_array(img))
            labels.append(label)
    return np.array(images, dtype=np.float32), np.array(labels)

images, labels = load_data(train_dir, IMG_SIZE)
print(f"Загружено изображений: {len(images)}, форма: {images.shape}")

# 4. Подготовка данных к моделированию

In [ ]:
# Нормализация данных (приведение значений пикселей к диапазону [0, 1]).
# Важно: делим float32, а не float64 — иначе массив раздувается и ядро падает.
images = images / np.float32(255.0)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    images, labels, test_size=0.2, random_state=42, stratify=labels
)
print(f"Train: {X_train.shape}, Test: {X_test.shape}")

# 5. Построение модели

CNN — Convolutional Neural Network — свёрточная нейронная сеть

In [ ]:
model = Sequential([
    Input(shape=(IMG_SIZE, IMG_SIZE, 3)),
    RandomFlip("horizontal"),
    RandomRotation(0.05),
    RandomZoom(0.1),

    Conv2D(32, (3, 3), activation="relu"),
    MaxPooling2D(pool_size=(2, 2)),

    Conv2D(64, (3, 3), activation="relu"),
    MaxPooling2D(pool_size=(2, 2)),

    Conv2D(128, (3, 3), activation="relu"),
    MaxPooling2D(pool_size=(2, 2)),

    Flatten(),
    Dense(128, activation="relu"),
    Dropout(0.5),
    Dense(len(categories), activation="softmax"),
])
model.summary()

# 6. Компиляция модели

In [ ]:
model.compile(
    optimizer=Adam(learning_rate=0.001),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)

# 7. Аугментация данных

In [ ]:
# Аугментация уже в модели: RandomFlip, RandomRotation, RandomZoom.
# Слои срабатывают только при обучении, на тесте идут исходные фото.

# 8. Обучение модели

In [ ]:
history = model.fit(
    X_train, y_train,
    validation_data=(X_test, y_test),
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    verbose=1,
)

# 9. Оценка модели

In [ ]:
test_loss, test_accuracy = model.evaluate(X_test, y_test, verbose=0)
print(f"Точность на тестовой выборке: {test_accuracy * 100:.2f}%")

# 10. Визуализация результатов

In [ ]:
plt.plot(history.history["accuracy"], label="Точность на обучающей выборке")
plt.plot(history.history["val_accuracy"], label="Точность на тестовой выборке")
plt.xlabel("Эпохи")
plt.ylabel("Точность")
plt.legend()
plt.show()

# 11. Тестирование работы модели

In [ ]:
sample_image = X_test[0].reshape(1, IMG_SIZE, IMG_SIZE, 3)
prediction = model.predict(sample_image, verbose=0)
predicted_class = categories[int(np.argmax(prediction))]
true_class = categories[int(y_test[0])]

print(f"Настоящий класс: {true_class}")
print(f"Предсказанный класс: {predicted_class}")

plt.imshow(np.clip(X_test[0], 0, 1))
plt.title(f"{true_class} → {predicted_class}")
plt.axis("off")
plt.show()